In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import os

def detect_changes(image1_path, image2_path, model_path, confidence_threshold=0.5, iou_threshold=0.5):
    model = YOLO(model_path)
    
    image1 = cv2.imread(image1_path)
    image2 = cv2.imread(image2_path)
    
    results1 = model(image1)[0]
    results2 = model(image2)[0]
    
    objects1 = extract_objects(results1, confidence_threshold)
    objects2 = extract_objects(results2, confidence_threshold)
    
    added, removed = compare_objects(objects1, objects2, iou_threshold)
    
    visualize_changes(image1, image2, added, removed)
    
    analyze_results(objects1, objects2, added, removed)

def extract_objects(results, confidence_threshold):
    objects = []
    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = r
        if score > confidence_threshold:
            objects.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'score': score,
                'class_id': int(class_id)
            })
    return objects

def compare_objects(objects1, objects2, iou_threshold):
    added = []
    removed = []
    
    matched1 = set()
    matched2 = set()
    
    for i, obj1 in enumerate(objects1):
        for j, obj2 in enumerate(objects2):
            if i in matched1 or j in matched2:
                continue
            
            iou = calculate_iou(obj1['bbox'], obj2['bbox'])
            if iou > iou_threshold:
                matched1.add(i)
                matched2.add(j)
                break
        
        if i not in matched1:
            removed.append(obj1)
    
    for j, obj2 in enumerate(objects2):
        if j not in matched2:
            added.append(obj2)
    
    return added, removed

def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    iou = intersection / float(area1 + area2 - intersection)
    return iou
    pass

def visualize_changes(image1, image2, added, removed):
    result_image = np.hstack((image1, image2))
    height, width = image1.shape[:2]
    
    for obj in added:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1 + width, y1), (x2 + width, y2), (0, 255, 0), 2)
        cv2.putText(result_image, 'Added', (x1 + width, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    for obj in removed:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(result_image, 'Removed', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    
    cv2.imshow('Changes Detected', result_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    pass

def analyze_results(objects1, objects2, added, removed):
    print(f"Image 1: {len(objects1)} objects detected")
    print(f"Image 2: {len(objects2)} objects detected")
    print(f"Added objects: {len(added)}")
    print(f"Removed objects: {len(removed)}")
    
    class_counts1 = {}
    class_counts2 = {}
    
    for obj in objects1:
        class_counts1[obj['class_id']] = class_counts1.get(obj['class_id'], 0) + 1
    
    for obj in objects2:
        class_counts2[obj['class_id']] = class_counts2.get(obj['class_id'], 0) + 1
    
    print("\nClass distribution in Image 1:")
    for class_id, count in class_counts1.items():
        print(f"Class {class_id}: {count}")
    
    print("\nClass distribution in Image 2:")
    for class_id, count in class_counts2.items():
        print(f"Class {class_id}: {count}")

def process_directory(directory_path, model_path):
    image_files = [f for f in os.listdir(directory_path) if f.endswith(('.jpg', '.png', '.jpeg'))]
    image_files.sort()
    
    for i in range(len(image_files) - 1):
        image1_path = os.path.join(directory_path, image_files[i])
        image2_path = os.path.join(directory_path, image_files[i+1])
        
        print(f"\nProcessing: {image_files[i]} and {image_files[i+1]}")
        detect_changes(image1_path, image2_path, model_path)

# 사용 예시
image1_path = 'C:/Users/AI-LHJ/Downloads/source/ch5/test 3.jpg'
image2_path = 'C:/Users/AI-LHJ/Downloads/source/ch5/test 4.jpg'
model_path = 'C:/Users/AI-LHJ/Desktop/ultralytics-main/runs/segment/train5/weights/best.pt'

detect_changes(image1_path, image2_path, model_path)


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import os

def detect_changes(image1_path, image2_path, model_path, confidence_threshold=0.5, iou_threshold=0.5):
    # YOLO 모델 로드
    model = YOLO(model_path)
    
    # 이미지 로드
    image1 = cv2.imread(image1_path)
    image2 = cv2.imread(image2_path)
    
    # YOLO를 사용하여 객체 탐지
    results1 = model(image1)[0]
    results2 = model(image2)[0]
    
    # 객체 추출
    objects1 = extract_objects(results1, confidence_threshold)
    objects2 = extract_objects(results2, confidence_threshold)
    
    # 객체 비교
    added, removed = compare_objects(objects1, objects2, iou_threshold)
    
    # 결과 시각화
    visualize_changes(image1, image2, added, removed)
    
    # 결과 분석
    analyze_results(objects1, objects2, added, removed)

def extract_objects(results, confidence_threshold):
    objects = []
    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = r
        if score > confidence_threshold:
            objects.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'score': score,
                'class_id': int(class_id)
            })
    return objects

def compare_objects(objects1, objects2, iou_threshold):
    added = []
    removed = []
    
    matched1 = set()
    matched2 = set()
    
    for i, obj1 in enumerate(objects1):
        for j, obj2 in enumerate(objects2):
            if i in matched1 or j in matched2:
                continue
            
            iou = calculate_iou(obj1['bbox'], obj2['bbox'])
            if iou > iou_threshold:
                matched1.add(i)
                matched2.add(j)
                break
        
        if i not in matched1:
            removed.append(obj1)
    
    for j, obj2 in enumerate(objects2):
        if j not in matched2:
            added.append(obj2)
    
    return added, removed

def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    iou = intersection / float(area1 + area2 - intersection)
    return iou

def visualize_changes(image1, image2, added, removed):
    # 두 이미지를 나란히 붙임
    result_image = np.hstack((image1, image2))
    height, width = image1.shape[:2]
    
    # 추가된 객체 표시 (녹색)
    for obj in added:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1 + width, y1), (x2 + width, y2), (0, 255, 0), 2)
        cv2.putText(result_image, 'Added', (x1 + width, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    # 제거된 객체 표시 (빨간색)
    for obj in removed:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(result_image, 'Removed', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    
    # 결과 표시
    cv2.imshow('Changes Detected', result_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

def analyze_results(objects1, objects2, added, removed):
    print(f"Image 1: {len(objects1)} objects detected")
    print(f"Image 2: {len(objects2)} objects detected")
    print(f"Added objects: {len(added)}")
    print(f"Removed objects: {len(removed)}")
    
    class_counts1 = {}
    class_counts2 = {}
    
    for obj in objects1:
        class_counts1[obj['class_id']] = class_counts1.get(obj['class_id'], 0) + 1
    
    for obj in objects2:
        class_counts2[obj['class_id']] = class_counts2.get(obj['class_id'], 0) + 1
    
    print("\nClass distribution in Image 1:")
    for class_id, count in class_counts1.items():
        print(f"Class {class_id}: {count}")
    
    print("\nClass distribution in Image 2:")
    for class_id, count in class_counts2.items():
        print(f"Class {class_id}: {count}")

# 사용 예시
image1_path = 'C:/Users/AI-LHJ/Downloads/source/ch5/test 1.jpg'
image2_path = 'C:/Users/AI-LHJ/Downloads/source/ch5/test 2.jpg'
model_path = 'C:/Users/AI-LHJ/Desktop/ultralytics-main/runs/segment/train8/weights/best.pt'

detect_changes(image1_path, image2_path, model_path)


In [ ]:
# 2D CNN
import cv2
import numpy as np
import torch
import torch.nn as nn
from ultralytics import YOLO

# 2D CNN 모델 정의
class Simple2DCNN(nn.Module):
    def __init__(self):
        super(Simple2DCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(64 * 160 * 160, 512)  # 640x640 크기에 맞게 조정
        self.fc2 = nn.Linear(512, 2)  # 예시: 클래스가 2개 (변화 감지, 비변화)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = x.view(-1, 64 * 160 * 160)  # Flatten
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# 변화 감지 함수
def detect_changes_2d(image1_path, image2_path, model_path, cnn_model, confidence_threshold=0.5, iou_threshold=0.5):
    # YOLO 모델 로드
    model = YOLO(model_path)

    # 이미지 로드
    image1 = cv2.imread(image1_path)
    image2 = cv2.imread(image2_path)

    # YOLO를 사용하여 객체 탐지
    results1 = model(image1)[0]
    results2 = model(image2)[0]

    # 객체 추출
    objects1 = extract_objects(results1, confidence_threshold)
    objects2 = extract_objects(results2, confidence_threshold)

    # 객체 비교 (기존 방식 유지)
    added, removed = compare_objects(objects1, objects2, iou_threshold)

    # 이미지를 2D CNN에 입력
    image1_2d = prepare_image_for_2d_cnn(image1)
    image2_2d = prepare_image_for_2d_cnn(image2)

    # 2D CNN을 통한 특징 추출 및 변화 감지
    features1 = cnn_model(image1_2d)
    features2 = cnn_model(image2_2d)
    
    difference = compute_difference_2d(features1, features2)

    # 결과 시각화
    visualize_changes(image1, image2, added, removed)

    # 결과 분석
    analyze_results(objects1, objects2, added, removed)

# 2D CNN에 입력할 이미지를 준비하는 함수
def prepare_image_for_2d_cnn(image):
    # 이미지를 2D CNN에 입력할 수 있도록 변환
    image = cv2.resize(image, (640, 640))
    image_2d = np.transpose(image, (2, 0, 1))  # (H, W, C) -> (C, H, W)
    return torch.Tensor(image_2d).unsqueeze(0)  # (B, C, H, W)

# 2D CNN을 통한 특징 비교 함수
def compute_difference_2d(features1, features2):
    # 간단한 유클리드 거리 계산
    return torch.norm(features1 - features2, p=2).item()

# 나머지 기존 함수들 유지 (extract_objects, compare_objects, calculate_iou, visualize_changes, analyze_results)

# CNN 모델 초기화
cnn_model = Simple2DCNN()

# 사용 예시
image1_path = 'C:/Users/AI-LHJ/Downloads/source/ch5/test 7.jpg'
image2_path = 'C:/Users/AI-LHJ/Downloads/source/ch5/test 8.jpg'
model_path = 'C:/Users/AI-LHJ/Desktop/ultralytics-main/runs/segment/train8/weights/best.pt'

detect_changes_2d(image1_path, image2_path, model_path, cnn_model)


In [ ]:
# 2D CNN
import cv2
import numpy as np
import torch
import torch.nn as nn
from ultralytics import YOLO

# 2D CNN 모델 정의
class Simple2DCNN(nn.Module):
    def __init__(self):
        super(Simple2DCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(64 * 160 * 160, 512)  # 640x640 크기에 맞게 조정
        self.fc2 = nn.Linear(512, 2)  # 예시: 클래스가 2개 (변화 감지, 비변화)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = x.view(-1, 64 * 160 * 160)  # Flatten
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# 변화 감지 함수
def detect_changes_2d(image1_path, image2_path, model_path, cnn_model, confidence_threshold=0.5, iou_threshold=0.5):
    # YOLO 모델 로드
    model = YOLO(model_path)

    # 이미지 로드
    image1 = cv2.imread(image1_path)
    image2 = cv2.imread(image2_path)

    # YOLO를 사용하여 객체 탐지
    results1 = model(image1)[0]
    results2 = model(image2)[0]

    # 객체 추출
    objects1 = extract_objects(results1, confidence_threshold)
    objects2 = extract_objects(results2, confidence_threshold)

    # 객체 비교 (기존 방식 유지)
    added, removed = compare_objects(objects1, objects2, iou_threshold)

    # 이미지를 2D CNN에 입력
    image1_2d = prepare_image_for_2d_cnn(image1)
    image2_2d = prepare_image_for_2d_cnn(image2)

    # 2D CNN을 통한 특징 추출 및 변화 감지
    features1 = cnn_model(image1_2d)
    features2 = cnn_model(image2_2d)
    
    difference = compute_difference_2d(features1, features2)

    # 결과 시각화
    visualize_changes(image1, image2, added, removed)

    # 결과 분석
    analyze_results(objects1, objects2, added, removed)

# 2D CNN에 입력할 이미지를 준비하는 함수
def prepare_image_for_2d_cnn(image):
    # 이미지를 2D CNN에 입력할 수 있도록 변환
    image = cv2.resize(image, (640, 640))
    image_2d = np.transpose(image, (2, 0, 1))  # (H, W, C) -> (C, H, W)
    return torch.Tensor(image_2d).unsqueeze(0)  # (B, C, H, W)

# 2D CNN을 통한 특징 비교 함수
def compute_difference_2d(features1, features2):
    # 간단한 유클리드 거리 계산
    return torch.norm(features1 - features2, p=2).item()

# 나머지 기존 함수들 유지 (extract_objects, compare_objects, calculate_iou, visualize_changes, analyze_results)
def extract_objects(results, confidence_threshold):
    objects = []
    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = r
        if score > confidence_threshold:
            objects.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'score': score,
                'class_id': int(class_id)
            })
    return objects

def track_objects_with_optical_flow(image1, image2, objects1):
    # 이전 프레임에서의 특징점 추출
    old_points = np.array([[(obj['bbox'][0] + obj['bbox'][2]) // 2, 
                            (obj['bbox'][1] + obj['bbox'][3]) // 2] for obj in objects1], dtype=np.float32)
    
    # Optical Flow 계산
    new_points, status, err = cv2.calcOpticalFlowPyrLK(cv2.cvtColor(image1, cv2.COLOR_BGR2GRAY), 
                                                       cv2.cvtColor(image2, cv2.COLOR_BGR2GRAY), 
                                                       old_points, None)
    
    return new_points, status, err

def compare_objects(objects1, objects2, iou_threshold):
    added = []
    removed = []
    
    matched1 = set()
    matched2 = set()
    
    for i, obj1 in enumerate(objects1):
        for j, obj2 in enumerate(objects2):
            if i in matched1 or j in matched2:
                continue
            
            iou = calculate_iou(obj1['bbox'], obj2['bbox'])
            if iou > iou_threshold:
                matched1.add(i)
                matched2.add(j)
                break
        
        if i not in matched1:
            removed.append(obj1)
    
    for j, obj2 in enumerate(objects2):
        if j not in matched2:
            added.append(obj2)
    
    return added, removed

def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    iou = intersection / float(area1 + area2 - intersection)
    return iou

def visualize_changes(image1, image2, added, removed, track_points=None):
    # 두 이미지를 나란히 붙임
    result_image = np.hstack((image1, image2))
    height, width = image1.shape[:2]
    
    # Optical Flow로 추적된 객체 표시 (파란색)
    if track_points is not None:
        for pt in track_points:
            if pt is not None:
                x, y = pt.ravel()
                cv2.circle(result_image, (int(x) + width, int(y)), 5, (255, 0, 0), -1)
    
    # 추가된 객체 표시 (녹색)
    for obj in added:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1 + width, y1), (x2 + width, y2), (0, 255, 0), 2)
        cv2.putText(result_image, 'Added', (x1 + width, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    # 제거된 객체 표시 (빨간색)
    for obj in removed:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(result_image, 'Removed', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    
    # 결과 표시
    cv2.imshow('Changes Detected', result_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

def analyze_results(objects1, objects2, added, removed):
    print(f"Image 1: {len(objects1)} objects detected")
    print(f"Image 2: {len(objects2)} objects detected")
    print(f"Added objects: {len(added)}")
    print(f"Removed objects: {len(removed)}")
    
    class_counts1 = {}
    class_counts2 = {}
    
    for obj in objects1:
        class_counts1[obj['class_id']] = class_counts1.get(obj['class_id'], 0) + 1
    
    for obj in objects2:
        class_counts2[obj['class_id']] = class_counts2.get(obj['class_id'], 0) + 1
    
    print("\nClass distribution in Image 1:")
    for class_id, count in class_counts1.items():
        print(f"Class {class_id}: {count}")
    
    print("\nClass distribution in Image 2:")
    for class_id, count in class_counts2.items():
        print(f"Class {class_id}: {count}")

# CNN 모델 초기화
cnn_model = Simple2DCNN()

# 사용 예시
image1_path = 'C:/Users/AI-LHJ/Downloads/source/ch5/test 1.jpg'
image2_path = 'C:/Users/AI-LHJ/Downloads/source/ch5/test 2.jpg'
model_path = 'C:/Users/AI-LHJ/Desktop/ultralytics-main/runs/segment/train8/weights/best.pt'

detect_changes_2d(image1_path, image2_path, model_path, cnn_model)


In [ ]:
## Optical Flow

import cv2
import numpy as np
from ultralytics import YOLO
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

def detect_changes_with_optical_flow(image1_path, image2_path, model_path, confidence_threshold=0.5, iou_threshold=0.5):
    # YOLO 모델 로드
    model = YOLO(model_path)
    
    # 이미지 로드
    image1 = cv2.imread(image1_path)
    image2 = cv2.imread(image2_path)
    
    # YOLO를 사용하여 객체 탐지
    results1 = model(image1)[0]
    results2 = model(image2)[0]
    
    # 객체 추출
    objects1 = extract_objects(results1, confidence_threshold)
    objects2 = extract_objects(results2, confidence_threshold)
    
    # Optical Flow를 사용한 객체 이동 추적
    track_points, status, err = track_objects_with_optical_flow(image1, image2, objects1)
    
    # 객체 비교
    added, removed = compare_objects(objects1, objects2, iou_threshold)
    
    # 결과 시각화
    visualize_changes(image1, image2, added, removed, track_points)
    
    # 결과 분석
    analyze_results(objects1, objects2, added, removed)

def extract_objects(results, confidence_threshold):
    objects = []
    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = r
        if score > confidence_threshold:
            objects.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'score': score,
                'class_id': int(class_id)
            })
    return objects

def track_objects_with_optical_flow(image1, image2, objects1):
    # 이전 프레임에서의 특징점 추출
    old_points = np.array([[(obj['bbox'][0] + obj['bbox'][2]) // 2, 
                            (obj['bbox'][1] + obj['bbox'][3]) // 2] for obj in objects1], dtype=np.float32)
    
    # Optical Flow 계산
    new_points, status, err = cv2.calcOpticalFlowPyrLK(cv2.cvtColor(image1, cv2.COLOR_BGR2GRAY), 
                                                       cv2.cvtColor(image2, cv2.COLOR_BGR2GRAY), 
                                                       old_points, None)
    
    return new_points, status, err

def compare_objects(objects1, objects2, iou_threshold):
    added = []
    removed = []
    
    matched1 = set()
    matched2 = set()
    
    for i, obj1 in enumerate(objects1):
        for j, obj2 in enumerate(objects2):
            if i in matched1 or j in matched2:
                continue
            
            iou = calculate_iou(obj1['bbox'], obj2['bbox'])
            if iou > iou_threshold:
                matched1.add(i)
                matched2.add(j)
                break
        
        if i not in matched1:
            removed.append(obj1)
    
    for j, obj2 in enumerate(objects2):
        if j not in matched2:
            added.append(obj2)
    
    return added, removed

def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    iou = intersection / float(area1 + area2 - intersection)
    return iou

def visualize_changes(image1, image2, added, removed, track_points=None):
    # 두 이미지를 나란히 붙임
    result_image = np.hstack((image1, image2))
    height, width = image1.shape[:2]
    
    # Optical Flow로 추적된 객체 표시 (파란색)
    if track_points is not None:
        for pt in track_points:
            if pt is not None:
                x, y = pt.ravel()
                cv2.circle(result_image, (int(x) + width, int(y)), 5, (255, 0, 0), -1)
    
    # 추가된 객체 표시 (녹색)
    for obj in added:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1 + width, y1), (x2 + width, y2), (0, 255, 0), 2)
        cv2.putText(result_image, 'Added', (x1 + width, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    # 제거된 객체 표시 (빨간색)
    for obj in removed:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(result_image, 'Removed', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    
    # 결과 표시
    cv2.imshow('Changes Detected', result_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

def analyze_results(objects1, objects2, added, removed):
    print(f"Image 1: {len(objects1)} objects detected")
    print(f"Image 2: {len(objects2)} objects detected")
    print(f"Added objects: {len(added)}")
    print(f"Removed objects: {len(removed)}")
    
    class_counts1 = {}
    class_counts2 = {}
    
    for obj in objects1:
        class_counts1[obj['class_id']] = class_counts1.get(obj['class_id'], 0) + 1
    
    for obj in objects2:
        class_counts2[obj['class_id']] = class_counts2.get(obj['class_id'], 0) + 1
    
    print("\nClass distribution in Image 1:")
    for class_id, count in class_counts1.items():
        print(f"Class {class_id}: {count}")
    
    print("\nClass distribution in Image 2:")
    for class_id, count in class_counts2.items():
        print(f"Class {class_id}: {count}")

# 사용 예시
image1_path = 'D:/LHJ-Capstone Wire detection/wireod/test (5).jpg'
image2_path = 'D:/LHJ-Capstone Wire detection/wireod/test (6).jpg'
model_path = 'D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weights/best.pt'

detect_changes_with_optical_flow(image1_path, image2_path, model_path)


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt

# 객체 추출 함수
def extract_objects(results, confidence_threshold):
    # Ultralytics YOLOv8 API 기준
    if hasattr(results, "boxes"):
        boxes = results.boxes.data.tolist()
    elif isinstance(results, list) and hasattr(results[0], "boxes"):
        boxes = results[0].boxes.data.tolist()
    else:
        boxes = []
    objects = []
    for r in boxes:
        x1, y1, x2, y2, score, class_id = r
        if score > confidence_threshold:
            objects.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'score': score,
                'class_id': int(class_id)
            })
    return objects

# IoU 계산 함수
def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    if union == 0:
        return 0.0
    iou = intersection / union
    return iou

# 객체 비교 함수
def compare_objects(objects1, objects2, iou_threshold):
    added = []
    removed = []
    matched1 = set()
    matched2 = set()
    for i, obj1 in enumerate(objects1):
        for j, obj2 in enumerate(objects2):
            if i in matched1 or j in matched2:
                continue
            iou = calculate_iou(obj1['bbox'], obj2['bbox'])
            if iou > iou_threshold:
                matched1.add(i)
                matched2.add(j)
                break
        if i not in matched1:
            removed.append(obj1)
    for j, obj2 in enumerate(objects2):
        if j not in matched2:
            added.append(obj2)
    return added, removed

# Optical Flow 추적 함수
def track_objects_with_optical_flow(image1, image2, objects1):
    # 중심점만 추적
    old_points = np.array(
        [[(obj['bbox'][0] + obj['bbox'][2]) // 2, (obj['bbox'][1] + obj['bbox'][3]) // 2] for obj in objects1],
        dtype=np.float32
    ).reshape(-1, 1, 2)
    if len(old_points) == 0:
        return None, None, None
    new_points, status, err = cv2.calcOpticalFlowPyrLK(
        cv2.cvtColor(image1, cv2.COLOR_BGR2GRAY),
        cv2.cvtColor(image2, cv2.COLOR_BGR2GRAY),
        old_points, None
    )
    return new_points, status, err

# 시각화 함수 (Matplotlib)
def visualize_changes(image1, image2, added, removed, track_points=None):
    result_image = np.hstack((image1, image2))
    height, width = image1.shape[:2]
    # Optical Flow 추적점 표시
    if track_points is not None:
        for pt in track_points:
            if pt is not None:
                x, y = pt.ravel()
                cv2.circle(result_image, (int(x) + width, int(y)), 5, (255, 0, 0), -1)
    # 추가 객체(녹색)
    for obj in added:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1 + width, y1), (x2 + width, y2), (0, 255, 0), 2)
        cv2.putText(result_image, 'Added', (x1 + width, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    # 제거 객체(빨간색)
    for obj in removed:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(result_image, 'Removed', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    plt.figure(figsize=(16, 8))
    plt.imshow(cv2.cvtColor(result_image, cv2.COLOR_BGR2RGB))
    plt.title('Changes Detected with Optical Flow')
    plt.axis('off')
    plt.show()

# 결과 분석 함수
def analyze_results(objects1, objects2, added, removed):
    print(f"Image 1: {len(objects1)} objects detected")
    print(f"Image 2: {len(objects2)} objects detected")
    print(f"Added objects: {len(added)}")
    print(f"Removed objects: {len(removed)}")
    class_counts1 = {}
    class_counts2 = {}
    for obj in objects1:
        class_counts1[obj['class_id']] = class_counts1.get(obj['class_id'], 0) + 1
    for obj in objects2:
        class_counts2[obj['class_id']] = class_counts2.get(obj['class_id'], 0) + 1
    print("\nClass distribution in Image 1:")
    for class_id, count in class_counts1.items():
        print(f"Class {class_id}: {count}")
    print("\nClass distribution in Image 2:")
    for class_id, count in class_counts2.items():
        print(f"Class {class_id}: {count}")

# 메인 함수
def detect_changes_with_optical_flow(image1_path, image2_path, model_path, confidence_threshold=0.5, iou_threshold=0.5):
    model = YOLO(model_path)
    image1 = cv2.imread(image1_path)
    image2 = cv2.imread(image2_path)
    results1 = model(image1)[0]
    results2 = model(image2)[0]
    objects1 = extract_objects(results1, confidence_threshold)
    objects2 = extract_objects(results2, confidence_threshold)
    track_points, status, err = track_objects_with_optical_flow(image1, image2, objects1)
    added, removed = compare_objects(objects1, objects2, iou_threshold)
    if track_points is not None:
        track_points = track_points.squeeze(axis=1)
    visualize_changes(image1, image2, added, removed, track_points)
    analyze_results(objects1, objects2, added, removed)

# 사용 예시
image1_path = 'D:/LHJ-Capstone Wire detection/wireod/test (5).jpg'
image2_path = 'D:/LHJ-Capstone Wire detection/wireod/test (6).jpg'
model_path = 'D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weights/best.pt'

detect_changes_with_optical_flow(image1_path, image2_path, model_path)


In [ ]:
#광류 기반 객체 추적 추가
import cv2
import numpy as np
from ultralytics import YOLO

def detect_objects(image_path, model_path, confidence_threshold=0.5):
    # YOLO 모델 로드
    model = YOLO(model_path)
    
    # 이미지 로드
    image = cv2.imread(image_path)
    
    # YOLO를 사용하여 객체 탐지
    results = model(image)[0]
    
    # 객체 추출
    objects = []
    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = r
        if score > confidence_threshold:
            objects.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'score': score,
                'class_id': int(class_id)
            })
    
    return objects

def compare_objects(objects1, objects2, iou_threshold=0.5):
    added = []
    removed = []
    
    matched1 = set()
    matched2 = set()
    
    for i, obj1 in enumerate(objects1):
        for j, obj2 in enumerate(objects2):
            if i in matched1 or j in matched2:
                continue
            
            iou = calculate_iou(obj1['bbox'], obj2['bbox'])
            if iou > iou_threshold:
                matched1.add(i)
                matched2.add(j)
                break
        
        if i not in matched1:
            removed.append(obj1)
    
    for j, obj2 in enumerate(objects2):
        if j not in matched2:
            added.append(obj2)
    
    return added, removed

def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    iou = intersection / float(area1 + area2 - intersection)
    return iou

def track_objects_with_optical_flow(image1, image2, objects1):
    old_points = np.array([[(obj['bbox'][0] + obj['bbox'][2]) // 2, 
                            (obj['bbox'][1] + obj['bbox'][3]) // 2] for obj in objects1], dtype=np.float32)
    
    new_points, status, err = cv2.calcOpticalFlowPyrLK(cv2.cvtColor(image1, cv2.COLOR_BGR2GRAY), 
                                                       cv2.cvtColor(image2, cv2.COLOR_BGR2GRAY), 
                                                       old_points, None)
    
    return new_points, status, err

def visualize_changes_with_optical_flow(image1_path, image2_path, added, removed, track_points):
    image1 = cv2.imread(image1_path)
    image2 = cv2.imread(image2_path)
    
    result_image = np.hstack((image1, image2))
    height, width = image1.shape[:2]
    
    for pt in track_points:
        if pt is not None:
            x, y = pt.ravel()
            cv2.circle(result_image, (int(x) + width, int(y)), 5, (255, 0, 0), -1)
    
    for obj in added:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1 + width, y1), (x2 + width, y2), (0, 255, 0), 2)
        cv2.putText(result_image, 'Added', (x1 + width, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    for obj in removed:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(result_image, 'Removed', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    
    cv2.imshow('Changes Detected with Optical Flow', result_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

def detect_changes_with_optical_flow(image1_path, image2_path, model_path, confidence_threshold=0.5, iou_threshold=0.5):
    # YOLO 모델을 사용하여 첫 번째 이미지의 객체 탐지
    image1_objects = detect_objects(image1_path, model_path, confidence_threshold)
    
    # YOLO 모델을 사용하여 두 번째 이미지의 객체 탐지
    image2_objects = detect_objects(image2_path, model_path, confidence_threshold)
    
    # 두 이미지 간의 객체 비교
    added, removed = compare_objects(image1_objects, image2_objects, iou_threshold)
    
    # 첫 번째 이미지에서 두 번째 이미지로의 객체 이동 추적 (Optical Flow)
    image1 = cv2.imread(image1_path)
    image2 = cv2.imread(image2_path)
    track_points, status, err = track_objects_with_optical_flow(image1, image2, image1_objects)
    
    # 변화 시각화 (Optical Flow 포함)
    visualize_changes_with_optical_flow(image1_path, image2_path, added, removed, track_points)

# 사용 예시
image1_path = 'D:/LHJ-Capstone Wire detection/wireod/test (5).jpg'
image2_path = 'D:/LHJ-Capstone Wire detection/wireod/test (6).jpg'
model_path = 'D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weights/best.pt'

detect_changes_with_optical_flow(image1_path, image2_path, model_path)


In [ ]:
#YOLO + IOU (2024.09.03)
import cv2
import numpy as np  # numpy 임포트 추가
from ultralytics import YOLO

def detect_objects(image_path, model_path, confidence_threshold=0.5):
    # YOLO 모델 로드
    model = YOLO(model_path)
    
    # 이미지 로드
    image = cv2.imread(image_path)
    
    # YOLO를 사용하여 객체 탐지
    results = model(image)[0]
    
    # 객체 추출
    objects = []
    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = r
        if score > confidence_threshold:
            objects.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'score': score,
                'class_id': int(class_id)
            })
    
    return objects

def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    iou = intersection / float(area1 + area2 - intersection)
    return iou

def compare_objects(objects1, objects2, iou_threshold=0.5):
    added = []
    removed = []
    
    matched1 = set()
    matched2 = set()
    
    for i, obj1 in enumerate(objects1):
        for j, obj2 in enumerate(objects2):
            if i in matched1 or j in matched2:
                continue
            
            iou = calculate_iou(obj1['bbox'], obj2['bbox'])
            if iou > iou_threshold:
                matched1.add(i)
                matched2.add(j)
                break
        
        if i not in matched1:
            removed.append(obj1)
    
    for j, obj2 in enumerate(objects2):
        if j not in matched2:
            added.append(obj2)
    
    return added, removed

def visualize_changes(image1, image2, added, removed):
    # 두 이미지를 나란히 붙임
    result_image = np.hstack((image1, image2))
    height, width = image1.shape[:2]
    
    # 추가된 객체 표시 (녹색)
    for obj in added:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1 + width, y1), (x2 + width, y2), (0, 255, 0), 2)
        cv2.putText(result_image, 'Added', (x1 + width, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    # 제거된 객체 표시 (빨간색)
    for obj in removed:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(result_image, 'Removed', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    
    # 결과 표시
    cv2.imshow('Changes Detected', result_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

# 사용 예시
image1_path = 'D:/LHJ-Capstone Wire detection/wireod/test (5).jpg'
image2_path = 'D:/LHJ-Capstone Wire detection/wireod/test (6).jpg'
model_path = 'D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weights/best.pt'

image1 = cv2.imread(image1_path)
image2 = cv2.imread(image2_path)

objects1 = detect_objects(image1_path, model_path)
objects2 = detect_objects(image2_path, model_path)

added, removed = compare_objects(objects1, objects2)

visualize_changes(image1, image2, added, removed)



In [ ]:
import cv2
import torch
import numpy as np
from ultralytics import YOLO
from PIL import Image
import torchvision.transforms as transforms

# YOLO 모델 로드
model_path = 'C:/Users/AI-LHJ/Desktop/ultralytics-main/runs/segment/train5/weights/best.pt'
model = YOLO(model_path)

# 객체 탐지 함수
def detect_objects(image, model, confidence_threshold=0.5):
    results = model(image)[0]
    objects = []
    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = r
        if score > confidence_threshold:
            objects.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'score': score,
                'class_id': int(class_id)
            })
    return objects

# CNN 기반 특징 추출기 정의
class FeatureExtractor(nn.Module):
    def __init__(self):
        super(FeatureExtractor, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.fc = nn.Linear(256 * 80 * 80, 512)  # 640x640 이미지의 최종 크기에 맞게 조정

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))  # 320x320
        x = self.pool(torch.relu(self.conv2(x)))  # 160x160
        x = self.pool(torch.relu(self.conv3(x)))  # 80x80
        x = x.view(-1, 256 * 80 * 80)  # Flatten
        x = torch.relu(self.fc(x))
        return x

# 두 이미지 간의 차이 계산
def compute_difference(image1, image2, model, feature_extractor):
    transform = transforms.Compose([
        transforms.Resize((640, 640)),
        transforms.ToTensor(),
    ])

    image1 = transform(image1).unsqueeze(0)
    image2 = transform(image2).unsqueeze(0)

    # 모델을 사용하여 특징 벡터 추출
    feature1 = feature_extractor(image1)
    feature2 = feature_extractor(image2)

    # 유클리드 거리로 두 특징 간의 차이를 계산
    difference = torch.norm(feature1 - feature2, p=2, dim=1)
    return difference.item()

# 객체 비교 및 시각화
def visualize_changes(image1, image2, added, removed):
    # 두 이미지를 나란히 붙임
    result_image = np.hstack((image1, image2))
    height, width = image1.shape[:2]
    
    # 추가된 객체 표시 (녹색)
    for obj in added:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1 + width, y1), (x2 + width, y2), (0, 255, 0), 2)
        cv2.putText(result_image, 'Added', (x1 + width, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    # 제거된 객체 표시 (빨간색)
    for obj in removed:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(result_image, 'Removed', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    
    # 결과 표시
    cv2.imshow('Changes Detected', result_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()



# 모델 초기화
feature_extractor = FeatureExtractor()

# 이미지 로드 및 객체 탐지
image1_path = 'C:/Users/AI-LHJ/Downloads/source/ch5/test 5.jpg'
image2_path = 'C:/Users/AI-LHJ/Downloads/source/ch5/test 6.jpg'

image1 = cv2.imread(image1_path)
image2 = cv2.imread(image2_path)

# YOLO를 사용한 객체 탐지
objects1 = detect_objects(image1, model)
objects2 = detect_objects(image2, model)

# PIL 이미지를 특징 추출에 사용할 수 있도록 변환
image1_pil = Image.fromarray(cv2.cvtColor(image1, cv2.COLOR_BGR2RGB))
image2_pil = Image.fromarray(cv2.cvtColor(image2, cv2.COLOR_BGR2RGB))

# 차이 계산
difference = compute_difference(image1_pil, image2_pil, model, feature_extractor)

# 두 이미지 간의 객체 변화 비교 (IoU 매칭이 추가되어야 함)
added = [obj for obj in objects2 if obj not in objects1]  # 간단한 비교
removed = [obj for obj in objects1 if obj not in objects2]

# 시각화
visualize_changes(image1, image2, added, removed)


In [ ]:
## last

import cv2
import torch
import numpy as np
from ultralytics import YOLO
import torch.nn as nn 
from PIL import Image
import torchvision.transforms as transforms

# YOLO 모델 로드
model_path = 'D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weights/best.pt'
model = YOLO(model_path)

# 객체 탐지 함수
def detect_objects(image, model, confidence_threshold=0.5):
    results = model(image)[0]
    objects = []
    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = r
        if score > confidence_threshold:
            objects.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'score': score,
                'class_id': int(class_id)
            })
    return objects

# IoU 계산 함수
def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])

    iou = intersection / float(area1 + area2 - intersection)
    return iou

# 객체 비교 및 변화된 객체 찾기 함수
def compare_objects_iou(objects1, objects2, iou_threshold=0.5):
    added = []
    removed = []

    matched1 = set()
    matched2 = set()

    for i, obj1 in enumerate(objects1):
        best_match_idx = None
        best_iou = 0
        for j, obj2 in enumerate(objects2):
            if j in matched2:
                continue

            iou = calculate_iou(obj1['bbox'], obj2['bbox'])
            if iou > best_iou:
                best_iou = iou
                best_match_idx = j

        if best_iou > iou_threshold:
            matched1.add(i)
            matched2.add(best_match_idx)
        else:
            removed.append(obj1)

    for j, obj2 in enumerate(objects2):
        if j not in matched2:
            added.append(obj2)

    return added, removed

# CNN 기반 특징 추출기 정의
class FeatureExtractor(nn.Module):
    def __init__(self):
        super(FeatureExtractor, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.fc = nn.Linear(256 * 80 * 80, 512)  # 640x640 이미지의 최종 크기에 맞게 조정

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))  # 320x320
        x = self.pool(torch.relu(self.conv2(x)))  # 160x160
        x = self.pool(torch.relu(self.conv3(x)))  # 80x80
        x = x.view(-1, 256 * 80 * 80)  # Flatten
        x = torch.relu(self.fc(x))
        return x

# 두 이미지 간의 차이 계산
def compute_difference(image1, image2, model, feature_extractor):
    transform = transforms.Compose([
        transforms.Resize((640, 640)),
        transforms.ToTensor(),
    ])

    image1 = transform(image1).unsqueeze(0)
    image2 = transform(image2).unsqueeze(0)

    # 모델을 사용하여 특징 벡터 추출
    feature1 = feature_extractor(image1)
    feature2 = feature_extractor(image2)

    # 유클리드 거리로 두 특징 간의 차이를 계산
    difference = torch.norm(feature1 - feature2, p=2, dim=1)
    return difference.item()

# 객체 비교 및 시각화
def visualize_changes(image1, image2, added, removed):
    # 두 이미지를 나란히 붙임
    result_image = np.hstack((image1, image2))
    height, width = image1.shape[:2]
    
    # 추가된 객체 표시 (녹색)
    for obj in added:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1 + width, y1), (x2 + width, y2), (0, 255, 0), 2)
        cv2.putText(result_image, 'Added', (x1 + width, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    # 제거된 객체 표시 (빨간색)
    for obj in removed:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(result_image, 'Removed', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    
    # 결과 표시
    cv2.imshow('Changes Detected', result_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

# 모델 초기화
feature_extractor = FeatureExtractor()

# 이미지 로드 및 객체 탐지
image1_path = 'D:/LHJ-Capstone Wire detection/wireod/test (5).jpg'
image2_path = 'D:/LHJ-Capstone Wire detection/wireod/test (6).jpg'


image1 = cv2.imread(image1_path)
image2 = cv2.imread(image2_path)

# YOLO를 사용한 객체 탐지
objects1 = detect_objects(image1, model)
objects2 = detect_objects(image2, model)

# 두 이미지 간의 객체 변화 비교 (IoU 매칭 적용)
added, removed = compare_objects_iou(objects1, objects2)

# 시각화
visualize_changes(image1, image2, added, removed)


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt

def detect_objects(image_path, model_path, confidence_threshold=0.5):
    # YOLO 모델 로드
    model = YOLO(model_path)
    
    # 이미지 로드
    image = cv2.imread(image_path)
    
    # YOLO를 사용하여 객체 탐지
    results = model(image, verbose=False)[0]
    if results.boxes is None:
        return []
    
    # 객체 추출
    objects = []
    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = r
        if score > confidence_threshold:
            objects.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'score': score,
                'class_id': int(class_id)
            })
    
    return objects

def extract_object_patches(image, objects):
    patches = []
    for obj in objects:
        x1, y1, x2, y2 = obj['bbox']
        # 유효한 영역만 추출
        if x2 > x1 and y2 > y1 and all(v >= 0 for v in [x1, y1, x2, y2]):
            patch = image[y1:y2, x1:x2]
            patches.append((patch, obj))
    return patches

def orb_feature_matching(patch1, patch2):
    # ORB 객체 생성
    orb = cv2.ORB_create(nfeatures=500)
    # 특징점 및 디스크립터 계산
    keypoints1, descriptors1 = orb.detectAndCompute(patch1, None)
    keypoints2, descriptors2 = orb.detectAndCompute(patch2, None)
    # 디스크립터가 None인 경우 처리
    if descriptors1 is None or descriptors2 is None:
        return [], keypoints1, keypoints2
    # BFMatcher 객체 생성 (Hamming 거리)
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    # 디스크립터 매칭
    matches = bf.match(descriptors1, descriptors2)
    # 매칭 결과 정렬
    matches = sorted(matches, key=lambda x: x.distance)
    return matches, keypoints1, keypoints2

def compare_objects_with_orb(image1, image2, objects1, objects2):
    added = []
    removed = []
    image1_patches = extract_object_patches(image1, objects1)
    image2_patches = extract_object_patches(image2, objects2)
    matched1 = set()
    matched2 = set()
    for i, (patch1, obj1) in enumerate(image1_patches):
        best_match = None
        best_match_idx = -1
        max_matches = 0
        for j, (patch2, obj2) in enumerate(image2_patches):
            if j in matched2:
                continue
            matches, _, _ = orb_feature_matching(patch1, patch2)
            if len(matches) > max_matches:
                max_matches = len(matches)
                best_match = obj2
                best_match_idx = j
        if best_match is not None and max_matches > 0:
            matched1.add(i)
            matched2.add(best_match_idx)
        else:
            removed.append(obj1)
    for j, obj2 in enumerate(objects2):
        if j not in matched2:
            added.append(obj2)
    return added, removed

def visualize_changes(image1, image2, added, removed):
    # 두 이미지를 나란히 붙임
    result_image = np.hstack((image1, image2))
    height, width = image1.shape[:2]
    # 추가된 객체 표시 (녹색)
    for obj in added:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1 + width, y1), (x2 + width, y2), (0, 255, 0), 2)
        cv2.putText(result_image, 'Added', (x1 + width, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    # 제거된 객체 표시 (빨간색)
    for obj in removed:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(result_image, 'Removed', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    # Matplotlib으로 결과 시각화
    plt.figure(figsize=(16, 8))
    plt.imshow(cv2.cvtColor(result_image, cv2.COLOR_BGR2RGB))
    plt.title('Changes Detected')
    plt.axis('off')
    plt.show()

def detect_changes_with_orb(image1_path, image2_path, model_path, confidence_threshold=0.5):
    # 첫 번째 이미지의 객체 탐지
    image1 = cv2.imread(image1_path)
    objects1 = detect_objects(image1_path, model_path, confidence_threshold)
    # 두 번째 이미지의 객체 탐지
    image2 = cv2.imread(image2_path)
    objects2 = detect_objects(image2_path, model_path, confidence_threshold)
    # ORB를 사용한 객체 비교
    added, removed = compare_objects_with_orb(image1, image2, objects1, objects2)
    # 변화 시각화
    visualize_changes(image1, image2, added, removed)

# 사용 예시
image1_path = 'D:/LHJ-Capstone Wire detection/wireod/test (5).jpg'
image2_path = 'D:/LHJ-Capstone Wire detection/wireod/test (6).jpg'
model_path = 'D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weights/best.pt'

detect_changes_with_orb(image1_path, image2_path, model_path)


In [ ]:
#SURF 알고리즘

import cv2
import numpy as np
from ultralytics import YOLO

def detect_objects(image_path, model_path, confidence_threshold=0.5):
    # YOLO 모델 로드
    model = YOLO(model_path)
    
    # 이미지 로드
    image = cv2.imread(image_path)
    
    # YOLO를 사용하여 객체 탐지
    results = model(image)[0]
    
    # 객체 추출
    objects = []
    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = r
        if score > confidence_threshold:
            objects.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'score': score,
                'class_id': int(class_id)
            })
    
    return objects

def extract_object_patches(image, objects):
    patches = []
    for obj in objects:
        x1, y1, x2, y2 = obj['bbox']
        patch = image[y1:y2, x1:x2]
        patches.append((patch, obj))
    return patches

def surf_feature_matching(patch1, patch2):
    # SURF 객체 생성 (임계값 400은 일반적으로 사용되는 값입니다)
    surf = cv2.xfeatures2d.SURF_create()
    
    # 특징점 및 디스크립터 계산
    keypoints1, descriptors1 = surf.detectAndCompute(patch1, None)
    keypoints2, descriptors2 = surf.detectAndCompute(patch2, None)
    
    # 디스크립터가 None인 경우 처리
    if descriptors1 is None or descriptors2 is None:
        return [], keypoints1, keypoints2
    
    # BFMatcher 객체 생성
    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
    
    # 디스크립터 매칭
    matches = bf.match(descriptors1, descriptors2)
    
    # 매칭 결과 정렬
    matches = sorted(matches, key=lambda x: x.distance)
    
    return matches, keypoints1, keypoints2

def compare_objects_with_surf(image1, image2, objects1, objects2):
    added = []
    removed = []
    
    image1_patches = extract_object_patches(image1, objects1)
    image2_patches = extract_object_patches(image2, objects2)
    
    matched1 = set()
    matched2 = set()
    
    for i, (patch1, obj1) in enumerate(image1_patches):
        best_match = None
        best_match_idx = -1
        max_matches = 0
        
        for j, (patch2, obj2) in enumerate(image2_patches):
            if j in matched2:
                continue
            
            matches, _, _ = surf_feature_matching(patch1, patch2)
            
            if len(matches) > max_matches:
                max_matches = len(matches)
                best_match = obj2
                best_match_idx = j
        
        if best_match is not None and max_matches > 0:
            matched1.add(i)
            matched2.add(best_match_idx)
        else:
            removed.append(obj1)
    
    for j, obj2 in enumerate(objects2):
        if j not in matched2:
            added.append(obj2)
    
    return added, removed

def visualize_changes(image1, image2, added, removed):
    # 두 이미지를 나란히 붙임
    result_image = np.hstack((image1, image2))
    height, width = image1.shape[:2]
    
    # 추가된 객체 표시 (녹색)
    for obj in added:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1 + width, y1), (x2 + width, y2), (0, 255, 0), 2)
        cv2.putText(result_image, 'Added', (x1 + width, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    # 제거된 객체 표시 (빨간색)
    for obj in removed:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(result_image, 'Removed', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    
    # 결과 표시
    cv2.imshow('Changes Detected', result_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

def detect_changes_with_surf(image1_path, image2_path, model_path, confidence_threshold=0.5):
    # YOLO 모델을 사용하여 첫 번째 이미지의 객체 탐지
    image1 = cv2.imread(image1_path)
    objects1 = detect_objects(image1_path, model_path, confidence_threshold)
    
    # YOLO 모델을 사용하여 두 번째 이미지의 객체 탐지
    image2 = cv2.imread(image2_path)
    objects2 = detect_objects(image2_path, model_path, confidence_threshold)
    
    # SURF를 사용한 객체 비교
    added, removed = compare_objects_with_surf(image1, image2, objects1, objects2)
    
    # 변화 시각화
    visualize_changes(image1, image2, added, removed)

# 사용 예시
image1_path = 'D:/LHJ-Capstone Wire detection/wireod/test (5).jpg'
image2_path = 'D:/LHJ-Capstone Wire detection/wireod/test (6).jpg'
model_path = 'D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weights/best.pt'

detect_changes_with_surf(image1_path, image2_path, model_path)


In [ ]:
#ORB

import cv2
import numpy as np
from ultralytics import YOLO

def detect_objects(image_path, model_path, confidence_threshold=0.5):
    # YOLO 모델 로드
    model = YOLO(model_path)
    
    # 이미지 로드
    image = cv2.imread(image_path)
    
    # YOLO를 사용하여 객체 탐지
    results = model(image)[0]
    
    # 객체 추출
    objects = []
    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = r
        if score > confidence_threshold:
            objects.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'score': score,
                'class_id': int(class_id)
            })
    
    return objects

def extract_object_patches(image, objects):
    patches = []
    for obj in objects:
        x1, y1, x2, y2 = obj['bbox']
        patch = image[y1:y2, x1:x2]
        patches.append((patch, obj))
    return patches

def orb_feature_matching(patch1, patch2):
    # ORB 객체 생성
    orb = cv2.ORB_create()
    
    # 특징점 및 디스크립터 계산
    keypoints1, descriptors1 = orb.detectAndCompute(patch1, None)
    keypoints2, descriptors2 = orb.detectAndCompute(patch2, None)
    
    # 디스크립터가 None인 경우 처리
    if descriptors1 is None or descriptors2 is None:
        return [], keypoints1, keypoints2
    
    # BFMatcher 객체 생성 (해밍 거리 사용)
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    
    # 디스크립터 매칭
    matches = bf.match(descriptors1, descriptors2)
    
    # 매칭 결과 정렬
    matches = sorted(matches, key=lambda x: x.distance)
    
    return matches, keypoints1, keypoints2

def compare_objects_with_orb(image1, image2, objects1, objects2):
    added = []
    removed = []
    
    image1_patches = extract_object_patches(image1, objects1)
    image2_patches = extract_object_patches(image2, objects2)
    
    matched1 = set()
    matched2 = set()
    
    for i, (patch1, obj1) in enumerate(image1_patches):
        best_match = None
        best_match_idx = -1
        max_matches = 0
        
        for j, (patch2, obj2) in enumerate(image2_patches):
            if j in matched2:
                continue
            
            matches, _, _ = orb_feature_matching(patch1, patch2)
            
            if len(matches) > max_matches:
                max_matches = len(matches)
                best_match = obj2
                best_match_idx = j
        
        if best_match is not None and max_matches > 0:
            matched1.add(i)
            matched2.add(best_match_idx)
        else:
            removed.append(obj1)
    
    for j, obj2 in enumerate(objects2):
        if j not in matched2:
            added.append(obj2)
    
    return added, removed

def visualize_changes(image1, image2, added, removed):
    # 두 이미지를 나란히 붙임
    result_image = np.hstack((image1, image2))
    height, width = image1.shape[:2]
    
    # 추가된 객체 표시 (녹색)
    for obj in added:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1 + width, y1), (x2 + width, y2), (0, 255, 0), 2)
        cv2.putText(result_image, 'Added', (x1 + width, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    # 제거된 객체 표시 (빨간색)
    for obj in removed:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(result_image, 'Removed', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    
    # 결과 표시
    cv2.imshow('Changes Detected', result_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

def detect_changes_with_orb(image1_path, image2_path, model_path, confidence_threshold=0.5):
    # YOLO 모델을 사용하여 첫 번째 이미지의 객체 탐지
    image1 = cv2.imread(image1_path)
    objects1 = detect_objects(image1_path, model_path, confidence_threshold)
    
    # YOLO 모델을 사용하여 두 번째 이미지의 객체 탐지
    image2 = cv2.imread(image2_path)
    objects2 = detect_objects(image2_path, model_path, confidence_threshold)
    
    # ORB를 사용한 객체 비교
    added, removed = compare_objects_with_orb(image1, image2, objects1, objects2)
    
    # 변화 시각화
    visualize_changes(image1, image2, added, removed)

# 사용 예시
image1_path = 'D:/LHJ-Capstone Wire detection/wireod/test (5).jpg'
image2_path = 'D:/LHJ-Capstone Wire detection/wireod/test (6).jpg'
model_path = 'D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weights/best.pt'

detect_changes_with_orb(image1_path, image2_path, model_path)


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

def detect_objects(image_path, model_path, confidence_threshold=0.5):
    # YOLO 모델 로드
    model = YOLO(model_path)
    
    # 이미지 로드
    image = cv2.imread(image_path)
    
    # YOLO를 사용하여 객체 탐지
    results = model(image)[0]
    
    # 객체 추출
    objects = []
    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = r
        if score > confidence_threshold:
            objects.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'score': score,
                'class_id': int(class_id)
            })
    
    return objects

def extract_object_patches(image, objects):
    patches = []
    for obj in objects:
        x1, y1, x2, y2 = obj['bbox']
        patch = image[y1:y2, x1:x2]
        patches.append((patch, obj))
    return patches

def surf_feature_matching(patch1, patch2):
    # SURF 객체 생성 (임계값 400은 일반적으로 사용되는 값입니다)
    surf = cv2.xfeatures2d.SURF_create(500)
    
    # 특징점 및 디스크립터 계산
    keypoints1, descriptors1 = surf.detectAndCompute(patch1, None)
    keypoints2, descriptors2 = surf.detectAndCompute(patch2, None)
    
    # 디스크립터가 None인 경우 처리
    if descriptors1 is None or descriptors2 is None:
        return [], keypoints1, keypoints2
    
    # BFMatcher 객체 생성
    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
    
    # 디스크립터 매칭
    matches = bf.match(descriptors1, descriptors2)
    
    # 매칭 결과 정렬
    matches = sorted(matches, key=lambda x: x.distance)
    
    return matches, keypoints1, keypoints2

def compare_objects_with_surf(image1, image2, objects1, objects2):
    added = []
    removed = []
    
    image1_patches = extract_object_patches(image1, objects1)
    image2_patches = extract_object_patches(image2, objects2)
    
    matched1 = set()
    matched2 = set()
    
    for i, (patch1, obj1) in enumerate(image1_patches):
        best_match = None
        best_match_idx = -1
        max_matches = 0
        
        for j, (patch2, obj2) in enumerate(image2_patches):
            if j in matched2:
                continue
            
            matches, _, _ = surf_feature_matching(patch1, patch2)
            
            if len(matches) > max_matches:
                max_matches = len(matches)
                best_match = obj2
                best_match_idx = j
        
        if best_match is not None and max_matches > 0:
            matched1.add(i)
            matched2.add(best_match_idx)
        else:
            removed.append(obj1)
    
    for j, obj2 in enumerate(objects2):
        if j not in matched2:
            added.append(obj2)
    
    return added, removed

def visualize_changes(image1, image2, added, removed):
    # 두 이미지를 나란히 붙임
    result_image = np.hstack((image1, image2))
    height, width = image1.shape[:2]
    
    # 추가된 객체 표시 (녹색)
    for obj in added:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1 + width, y1), (x2 + width, y2), (0, 255, 0), 2)
        cv2.putText(result_image, 'Added', (x1 + width, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    # 제거된 객체 표시 (빨간색)
    for obj in removed:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(result_image, 'Removed', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    
    # 결과 표시
    cv2.imshow('Changes Detected', result_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

def detect_changes_with_surf(image1_path, image2_path, model_path, confidence_threshold=0.5):
    # YOLO 모델을 사용하여 첫 번째 이미지의 객체 탐지
    image1 = cv2.imread(image1_path)
    objects1 = detect_objects(image1_path, model_path, confidence_threshold)
    
    # YOLO 모델을 사용하여 두 번째 이미지의 객체 탐지
    image2 = cv2.imread(image2_path)
    objects2 = detect_objects(image2_path, model_path, confidence_threshold)
    
    # SURF를 사용한 객체 비교
    added, removed = compare_objects_with_surf(image1, image2, objects1, objects2)
    
    # 변화 시각화
    visualize_changes(image1, image2, added, removed)

# 사용 예시
image1_path = 'D:/LHJ-Capstone Wire detection/wireod/test (5).jpg'
image2_path = 'D:/LHJ-Capstone Wire detection/wireod/test (6).jpg'
model_path = 'D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weights/best.pt'

detect_changes_with_surf(image1_path, image2_path, model_path)


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

def detect_and_visualize_wires_segmentation(image_path, model_path, confidence_threshold=0.5):
    # 모델 로드
    model = YOLO(model_path)
    
    # 이미지 로드
    image = cv2.imread(image_path)
    
    # 객체 탐지 및 세그멘테이션 수행
    results = model(image, stream=True)
    
    # 탐지된 와이어 시각화
    visualized_image = visualize_wires_segmentation(image, results, confidence_threshold)
    
    # 결과 표시
    cv2.imshow('Wire Segmentation', visualized_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    
    # 결과 저장 (선택사항)
    cv2.imwrite('wire_segmentation_result.jpg', visualized_image)

def visualize_wires_segmentation(image, results, confidence_threshold):
    visualized_image = image.copy()
    
    for r in results:
        masks = r.masks
        if masks is None:
            continue
        
        for i, mask in enumerate(masks.data):
            if r.boxes.conf[i] > confidence_threshold and r.boxes.cls[i] == 2:  # 클래스 ID 0이 와이어라고 가정
                # 마스크를 이미지 크기에 맞게 조정
                mask = mask.cpu().numpy()
                mask = cv2.resize(mask, (image.shape[1], image.shape[0]))
                
                # 마스크를 이진화
                mask = (mask > 0.5).astype(np.uint8)
                
                # 세그멘테이션 마스크에 색상 적용
                color_mask = np.zeros_like(image, dtype=np.uint8)
                color_mask[mask == 1] = [0, 255, 0]  # 녹색으로 와이어 표시
                
                # 원본 이미지와 색상 마스크 블렌딩
                alpha = 0.5  # 투명도 설정
                visualized_image = cv2.addWeighted(visualized_image, 1, color_mask, alpha, 0)
                
                # 신뢰도 점수 표시
                score = r.boxes.conf[i].item()
                label = f'Wire: {score:.2f}'
                cv2.putText(visualized_image, label, (int(r.boxes.xyxy[i][0]), int(r.boxes.xyxy[i][1]) - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    return visualized_image

# 사용 예시
image_path = 'D:/LHJ-Capstone Wire detection/wireod/test(4)'
model_path = 'C:/Users/AI-LHJ/Desktop/ultralytics-main/runs/segment/train8/weights/best.pt'

detect_and_visualize_wires_segmentation(image_path, model_path, confidence_threshold=0.5)

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

def detect_and_visualize_objects(image_path, model_path, confidence_threshold=0.5):
    # Load model
    model = YOLO(model_path)
    
    # Load image
    image = cv2.imread(image_path)
    
    # Perform object detection and segmentation
    results = model(image, stream=True)
    
    # Visualize detected objects
    visualized_image = visualize_objects(image, results, confidence_threshold)
    
    # Display results
    cv2.imshow('Wire, Polo, and Cross Arm Segmentation', visualized_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    
    # Save results (optional)
    cv2.imwrite('segmentation_result.jpg', visualized_image)

def visualize_objects(image, results, confidence_threshold):
    visualized_image = image.copy()
    
    class_colors = {
        0: [0, 0, 255],  # Blue for wire
        1: [255, 0, 0],  # Red for polo
        2: [0, 255, 0]   # Green for cross arm
    }
    
    class_names = {
        0: "Cross Arm",
        1: "Polo",
        2 : "Wire"
    }
    
    for r in results:
        masks = r.masks
        if masks is None:
            continue
        
        for i, mask in enumerate(masks.data):
            class_id = int(r.boxes.cls[i].item())
            if r.boxes.conf[i] > confidence_threshold and class_id in [0, 1, 2]:
                # Resize mask to match image size
                mask = mask.cpu().numpy()
                mask = cv2.resize(mask, (image.shape[1], image.shape[0]))
                
                # Binarize mask
                mask = (mask > 0.5).astype(np.uint8)
                
                # Apply color to segmentation mask
                color_mask = np.zeros_like(image, dtype=np.uint8)
                color_mask[mask == 1] = class_colors[class_id]
                
                # Blend original image with color mask
                alpha = 0.5  # Set transparency
                visualized_image = cv2.addWeighted(visualized_image, 1, color_mask, alpha, 0)
                
                # Display confidence score
                score = r.boxes.conf[i].item()
                label = f'{class_names[class_id]}: {score:.2f}'
                cv2.putText(visualized_image, label, (int(r.boxes.xyxy[i][0]), int(r.boxes.xyxy[i][1]) - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, class_colors[class_id], 2)
    
    return visualized_image

# Usage example
image_path = 'D:/LHJ-Capstone Wire detection/wireod/test(4)'
model_path = 'D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weightsbest.pt'

detect_and_visualize_objects(image_path, model_path, confidence_threshold=0.5)